In [ ]:
# ============================================
# 0. Imports, paths, and basic setup
# ============================================
import os
import numpy as np
import pandas as pd
import pandas.api.types
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")

# Resolve the project root by walking up until pyproject.toml is found.
# This makes the notebook portable regardless of where it is run from.
from pathlib import Path as _Path
PROJECT_ROOT = _Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = str(PROJECT_ROOT)
DATA_PATH   = os.path.join(PROJECT_ROOT, "data", "raw")   # raw CSVs live here
REPORT_PATH = os.path.join(PROJECT_ROOT, "results")       # plots saved here
os.makedirs(REPORT_PATH, exist_ok=True)

# ============================================
# 1. Load data and time-based split
# ============================================
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

# Sort by date_id to preserve temporal order
train = train.sort_values("date_id").reset_index(drop=True)

n = len(train)
train_end = int(0.7 * n)
val_end   = int(0.9 * n)

train_set = train.iloc[:train_end].reset_index(drop=True)
val_set   = train.iloc[train_end:val_end].reset_index(drop=True)
test_set  = train.iloc[val_end:].reset_index(drop=True)

print(f"Total samples: {n:,}")
print(f"Train set: {len(train_set):,} rows ({len(train_set)/n:.1%})")
print(f"Validation set: {len(val_set):,} rows ({len(val_set)/n:.1%})")
print(f"Test set: {len(test_set):,} rows ({len(test_set)/n:.1%})")

# ============================================
# 2. Column groups and helper functions
# ============================================
meta_cols = ["date_id", "is_scored"]

finance_cols = [
    "forward_returns",
    "risk_free_rate",
    "market_forward_excess_returns",
    "lagged_forward_returns",
    "lagged_risk_free_rate",
    "lagged_market_forward_excess_returns",
]

non_feature_cols = meta_cols + finance_cols

# Feature cluster prefix map
prefix_map = {
    "M":   "Market Dynamics",
    "E":   "Economic",
    "I":   "Interest Rate",
    "P":   "Price/Valuation",
    "V":   "Volatility",
    "S":   "Sentiment",
    "MOM": "Momentum",
    "D":   "Dummy",
}


def drop_dummy_cols(df, cols):
    """Drop columns whose values are all within {-1, 0, 1} (dummy/binary flags)."""
    clean_cols = []
    for c in cols:
        vals = df[c].dropna().unique()
        if set(vals).issubset({-1, 0, 1}):
            continue
        clean_cols.append(c)
    return clean_cols


def get_feature_cols(df):
    """Return all feature columns (i.e. everything except meta and finance columns)."""
    return [c for c in df.columns if c not in non_feature_cols]


def corr_with_target(df, target, features, method="pearson"):
    """Compute correlation of each feature with the target, sorted by |corr| descending."""
    corr_series = df[features + [target]].corr(method=method)[target]
    corr_series = corr_series.drop(target)
    corr_series = corr_series.sort_values(key=lambda x: x.abs(), ascending=False)
    return corr_series


def build_cluster_means(df, features, prefix_map, zscore=True):
    """Compute row-wise mean per feature cluster (optionally z-score normalised first)."""
    sub = df[features].copy()
    if zscore:
        sub = (sub - sub.mean()) / sub.std(ddof=0)

    cluster_mean_df = pd.DataFrame(index=df.index)
    for prefix, cat_name in prefix_map.items():
        if prefix == "MOM":
            cols = [c for c in features if c.startswith("MOM")]
        else:
            cols = [c for c in features if c.startswith(prefix) and not c.startswith("MOM")]
        if len(cols) == 0:
            continue
        cluster_mean_df[prefix] = sub[cols].mean(axis=1, skipna=True)
    return cluster_mean_df


# ============================================
# 3. Missing rate (top 30 bar chart)
# ============================================
# Compute missing rate on numeric columns only, excluding date_id
num_cols = train_set.select_dtypes(include=[np.number]).columns.tolist()
if "date_id" in num_cols:
    num_cols.remove("date_id")

missing_rate = train_set[num_cols].isnull().mean()
missing_rate = missing_rate.sort_values(ascending=False)

top30_missing = missing_rate.head(30)
print("\nTop 30 missing rate columns:")
print(top30_missing)

plt.figure(figsize=(10, 6))
sns.barplot(
    x=top30_missing.values,
    y=top30_missing.index,
    orient="h",
)
plt.xlabel("Missing rate")
plt.ylabel("Feature")
plt.title("Top 30 Features by Missing Rate (Train Set)")
plt.tight_layout()
missing_fig_path = os.path.join(REPORT_PATH, "eda_missing_top30_bar.png")
plt.savefig(missing_fig_path, dpi=200)
plt.close()
print("Saved:", missing_fig_path)

# ============================================
# 4. Outlier rate (IQR-based, top 30 bar chart)
# ============================================
def compute_outlier_rate_iqr(df, cols):
    """Compute IQR-based outlier rate for each column. Returns a sorted DataFrame."""
    outlier_rates = {}
    for col in cols:
        series = df[col].dropna()
        if series.empty:
            outlier_rates[col] = np.nan
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            outlier_rates[col] = 0.0
            continue
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask_outlier = (df[col] < lower) | (df[col] > upper)
        outlier_rates[col] = mask_outlier.mean()

    outlier_df = pd.DataFrame.from_dict(outlier_rates, orient="index", columns=["outlier_rate"])
    outlier_df = outlier_df.sort_values("outlier_rate", ascending=False)
    return outlier_df


# Exclude dummy columns before computing outlier rates
feature_cols = get_feature_cols(train_set)
feature_no_dummy = drop_dummy_cols(train_set, feature_cols)

outlier_df = compute_outlier_rate_iqr(train_set, feature_no_dummy)
top30_outlier = outlier_df.head(30)

print("\nTop 30 outlier rate columns:")
print(top30_outlier)

plt.figure(figsize=(10, 6))
sns.barplot(
    x=top30_outlier["outlier_rate"].values,
    y=top30_outlier.index,
    orient="h",
)
plt.xlabel("Outlier rate (IQR-based)")
plt.ylabel("Feature")
plt.title("Top 30 Features by Outlier Rate (Train Set)")
plt.tight_layout()
outlier_fig_path = os.path.join(REPORT_PATH, "eda_outlier_top30_bar.png")
plt.savefig(outlier_fig_path, dpi=200)
plt.close()
print("Saved:", outlier_fig_path)

# ============================================
# 5. Correlation with target (feature vs target, top 15)
# ============================================
target_col = "forward_returns"

# Only keep features with <= 70% missing rate to ensure reliable correlations
valid_corr_cols = missing_rate[missing_rate <= 0.7].index.tolist()

# Intersect with numeric feature columns
train_feature_cols = get_feature_cols(train_set)
corr_cols = [
    c for c in train_feature_cols
    if (c in valid_corr_cols) and pandas.api.types.is_numeric_dtype(train_set[c])
]

print(f"\nValid feature columns for correlation analysis (missing <= 70%): {len(corr_cols)}")

feature_target_corr = corr_with_target(
    train_set,
    target_col,
    corr_cols,
    method="pearson",
)

top15_corr = feature_target_corr.head(15)
print("\nTop 15 feature-target correlations (Pearson):")
print(top15_corr)

plt.figure(figsize=(10, 6))
sns.barplot(
    x=top15_corr.values,
    y=top15_corr.index,
    orient="h",
)
plt.xlabel("Pearson correlation with forward_returns")
plt.ylabel("Feature")
plt.title("Top 15 Features vs Target Correlation (Train Set)")
plt.tight_layout()
corr_feature_fig_path = os.path.join(REPORT_PATH, "eda_corr_feature_target_top15_bar.png")
plt.savefig(corr_feature_fig_path, dpi=200)
plt.close()
print("Saved:", corr_feature_fig_path)

# ============================================
# 6. Cluster-level means vs target correlation
# ============================================
cluster_means_train = build_cluster_means(train_set, corr_cols, prefix_map, zscore=True)

cluster_target_corr = (
    pd.concat([cluster_means_train, train_set[target_col]], axis=1)
    .corr(method="pearson")[target_col]
    .drop(target_col)
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

print("\nCluster-level correlations with target:")
print(cluster_target_corr)

# Map prefix keys to full cluster names for the chart labels
cluster_labels = [prefix_map.get(prefix, prefix) for prefix in cluster_target_corr.index]

plt.figure(figsize=(8, 5))
sns.barplot(
    x=cluster_target_corr.values,
    y=cluster_labels,
    orient="h",
)
plt.xlabel("Pearson correlation with forward_returns")
plt.ylabel("Cluster")
plt.title("Cluster-mean Features vs Target Correlation (Train Set)")
plt.tight_layout()
corr_cluster_fig_path = os.path.join(REPORT_PATH, "eda_corr_cluster_target_bar.png")
plt.savefig(corr_cluster_fig_path, dpi=200)
plt.close()
print("Saved:", corr_cluster_fig_path)

# ============================================
# 7. Feature-feature correlation heatmap
#    (restricted to top 30 features by |corr with target| for readability)
# ============================================
top30_for_matrix = feature_target_corr.head(30).index.tolist()
corr_matrix = train_set[top30_for_matrix].corr(method="pearson")

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    square=True,
    cbar_kws={"shrink": 0.6},
)
plt.title("Feature-Feature Correlation Matrix (Top 30 by |corr with target|)")
plt.tight_layout()
corr_matrix_fig_path = os.path.join(REPORT_PATH, "eda_corr_feature_feature_heatmap.png")
plt.savefig(corr_matrix_fig_path, dpi=200)
plt.close()
print("Saved:", corr_matrix_fig_path)

# ============================================
# 8. Distributions of key finance columns
# ============================================
finance_plot_cols = [
    "forward_returns",
    "risk_free_rate",
    "market_forward_excess_returns",
]

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

for i, col in enumerate(finance_plot_cols):
    ax = axes[i]
    if col not in train_set.columns:
        ax.set_title(f"{col} (not in data)")
        ax.axis("off")
        continue
    series = train_set[col].dropna()
    sns.histplot(series, bins=50, kde=True, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")

fig.suptitle("Distribution of Key Financial Columns (Train Set)", y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.92])

dist_fig_path = os.path.join(REPORT_PATH, "eda_finance_distributions.png")
fig.savefig(dist_fig_path, dpi=200)
plt.close(fig)
print("Saved:", dist_fig_path)
